In [ ]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# import kagglehub

# # Download latest version
# path = kagglehub.dataset_download("nih-chest-xrays/data")

# print("Path to dataset files:", path)

# Imports

In [ ]:
# tf.keras.backend.clear_session()

In [ ]:
import os
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import cv2

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    precision_recall_curve,
    roc_curve
)

from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.keras.backend.clear_session()
print("TensorFlow:", tf.__version__)


# Load metadata and connect images

In [ ]:
BASE_PATH = "/kaggle/input/datasets/organizations/nih-chest-xrays/data"
CSV_PATH = os.path.join(BASE_PATH, "Data_Entry_2017.csv")

image_paths = {}
for root, _, files in os.walk(BASE_PATH):
    for file in files:
        if file.lower().endswith(".png"):
            image_paths[file] = os.path.join(root, file)

print("Total images found:", len(image_paths))

df = pd.read_csv(CSV_PATH)
print("Metadata shape:", df.shape)
df.head()


In [ ]:
df = df.copy()

if "View Position" in df.columns:
    df = df[df["View Position"].isin(["PA", "AP"])].copy()

df = df[
    (df["Finding Labels"] == "No Finding") |
    (df["Finding Labels"].str.contains("Pneumonia", na=False))
].copy()

df["label"] = df["Finding Labels"].apply(lambda x: 1 if "Pneumonia" in x else 0)
df["path"] = df["Image Index"].map(image_paths)

keep_cols = ["Image Index", "Finding Labels", "label", "path"]
if "Patient ID" in df.columns:
    keep_cols.append("Patient ID")
if "View Position" in df.columns:
    keep_cols.append("View Position")

df = df[keep_cols].dropna(subset=["path"]).drop_duplicates().reset_index(drop=True)

print("Filtered dataset shape:", df.shape)
print(df["label"].value_counts())
df.head()


In [ ]:
df['Finding Labels'].value_counts()

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())

print("\nUnique patients:", df["Patient ID"].nunique() if "Patient ID" in df.columns else "Patient ID not found")
print("Unique images:", df["Image Index"].nunique())
print("\nClass distribution:")
print(df["label"].value_counts(normalize=True).rename("ratio"))


In [ ]:
import cv2
import matplotlib.pyplot as plt

sample = df.sample(5)

for _, row in sample.iterrows():
    img = cv2.imread(row["path"])
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    plt.imshow(img)
    plt.title("Pneumonia" if row["label"] == 1 else "Normal")
    plt.axis("off")
    plt.show()

# Data preprocessing

In [ ]:
print("Duplicates before:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print("Duplicates after:", df.duplicated().sum())

if "Patient ID" in df.columns:
    patient_level = df.groupby("Patient ID")["label"].max().value_counts()
    print("\nPatient-level label distribution:")
    print(patient_level)

print("\nImage-level label distribution:")
print(df["label"].value_counts())


# Class imbalance strategy

In [ ]:
df_model = df.copy().reset_index(drop=True)

print(df_model["label"].value_counts())
print(df_model.shape)

plt.figure(figsize=(6, 4))
sns.countplot(x=df_model["label"])
plt.xticks([0, 1], ["Normal", "Pneumonia"])
plt.title("Image-Level Class Distribution")
plt.show()


In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x=df_model["label"])
plt.xticks([0, 1], ["Normal", "Pneumonia"])
plt.title("Original Class Distribution")
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
samples = df_model.sample(6, random_state=42).reset_index(drop=True)

for i, ax in enumerate(axes.flat):
    img = cv2.imread(samples.loc[i, "path"])
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title("Pneumonia" if samples.loc[i, "label"] == 1 else "Normal")
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
sizes = []

for p in df_model["path"].sample(min(200, len(df_model)), random_state=42):
    img = cv2.imread(p)
    if img is not None:
        sizes.append(img.shape[:2])

sizes_df = pd.DataFrame(sizes, columns=["Height", "Width"])
sizes_df.describe()


In [ ]:
loss_fn = tf.keras.losses.BinaryFocalCrossentropy(gamma=2.0)

class_weights = {0: 1.0, 1: 2.0}

In [ ]:
assert "Patient ID" in df_model.columns, "Patient ID column is required for patient-level split."

patient_labels = df_model.groupby("Patient ID")["label"].max().reset_index()

train_pat, temp_pat = train_test_split(
    patient_labels,
    test_size=0.30,
    stratify=patient_labels["label"],
    random_state=SEED
)

val_pat, test_pat = train_test_split(
    temp_pat,
    test_size=0.50,
    stratify=temp_pat["label"],
    random_state=SEED
)

train_df = df_model[df_model["Patient ID"].isin(train_pat["Patient ID"])].copy()
val_df   = df_model[df_model["Patient ID"].isin(val_pat["Patient ID"])].copy()
test_df  = df_model[df_model["Patient ID"].isin(test_pat["Patient ID"])].copy()

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

print("\nTrain label counts:")
print(train_df["label"].value_counts())
print("\nValidation label counts:")
print(val_df["label"].value_counts())
print("\nTest label counts:")
print(test_df["label"].value_counts())

assert set(train_df["Patient ID"]).isdisjoint(set(val_df["Patient ID"]))
assert set(train_df["Patient ID"]).isdisjoint(set(test_df["Patient ID"]))
assert set(val_df["Patient ID"]).isdisjoint(set(test_df["Patient ID"]))
print("\nNo patient leakage detected.")


In [ ]:
classes = np.array([0, 1])

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_df["label"].values
)

class_weights = {
    0: float(weights[0]),
    1: float(weights[1])
}

print(class_weights)


In [ ]:
# =========================
# CLASS IMBALANCE REVIEW
# =========================
from collections import Counter
from imblearn.over_sampling import SMOTE
import seaborn as sns
import matplotlib.pyplot as plt

print("Original TRAIN distribution:")
print(Counter(train_df["label"]))

plt.figure(figsize=(6,4))
sns.countplot(x=train_df["label"])
plt.xticks([0,1], ["Normal", "Pneumonia"])
plt.title("Before SMOTE - Train Distribution")
plt.show()


# ==========================================================
# FEATURE PREP FOR SMOTE
# ==========================================================
# Since SMOTE cannot use image paths directly,
# we use structured metadata for balancing support
# (Age + Gender + View Position)
# ==========================================================

smote_train = train_df.copy()

# Gender
if "Patient Gender" in df.columns:
    gender_map = df[["Image Index", "Patient Gender"]].drop_duplicates()
    gender_map["Patient Gender"] = gender_map["Patient Gender"].map({"M":0, "F":1})
    smote_train = smote_train.merge(
        gender_map,
        on="Image Index",
        how="left"
    )
else:
    smote_train["Patient Gender"] = 0

# Age
if "Patient Age" in df.columns:
    age_map = df[["Image Index", "Patient Age"]].drop_duplicates()
    smote_train = smote_train.merge(
        age_map,
        on="Image Index",
        how="left"
    )
else:
    smote_train["Patient Age"] = 50

# View
smote_train["View Position"] = smote_train["View Position"].map({"PA":0, "AP":1})

# Fill
smote_train["Patient Age"] = smote_train["Patient Age"].fillna(smote_train["Patient Age"].median())
smote_train["Patient Gender"] = smote_train["Patient Gender"].fillna(0)

X_train_smote = smote_train[["Patient Age", "Patient Gender", "View Position"]]
y_train_smote = smote_train["label"]


# =========================
# APPLY SMOTE
# =========================
smote = SMOTE(
    sampling_strategy=0.5,   # minority becomes 50% of majority
    random_state=SEED,
    k_neighbors=3
)

X_resampled, y_resampled = smote.fit_resample(X_train_smote, y_train_smote)

print("\nAfter SMOTE:")
print(Counter(y_resampled))


# =========================
# VISUALIZE
# =========================
plt.figure(figsize=(6,4))
sns.countplot(x=y_resampled)
plt.xticks([0,1], ["Normal", "Pneumonia"])
plt.title("After SMOTE - Train Distribution")
plt.show()


# =========================
# BUILD BALANCED TRAIN DF
# =========================
normal_df = train_df[train_df["label"] == 0].copy()
pneumonia_df = train_df[train_df["label"] == 1].copy()

target_pneumonia = Counter(y_resampled)[1]

pneumonia_upsampled = pneumonia_df.sample(
    n=target_pneumonia,
    replace=True,
    random_state=SEED
)

train_df_balanced = pd.concat(
    [normal_df, pneumonia_upsampled],
    axis=0
).sample(frac=1, random_state=SEED).reset_index(drop=True)

print("\nFinal image-train distribution:")
print(train_df_balanced["label"].value_counts())

In [ ]:
IMG_SIZE = 224   
BATCH_SIZE = 16
AUTOTUNE = tf.data.AUTOTUNE

train_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.03),
    tf.keras.layers.RandomTranslation(0.03, 0.03),
    tf.keras.layers.RandomZoom(0.05, 0.05),
], name="train_augmentation")


def decode_image(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_png(image, channels=1)
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.image.grayscale_to_rgb(image)
    image = tf.cast(image, tf.float32)
    label = tf.cast(label, tf.float32)
    return image, label


def prepare_train(path, label):
    image, label = decode_image(path, label)
    image = train_augmentation(image, training=True)
    image = preprocess_input(image)
    return image, label


def prepare_eval(path, label):
    image, label = decode_image(path, label)
    image = preprocess_input(image)
    return image, label


def make_dataset(dataframe, training=False):
    ds = tf.data.Dataset.from_tensor_slices((
        dataframe["path"].values,
        dataframe["label"].values.astype("float32")
    ))

    if training:
        ds = ds.shuffle(len(dataframe), seed=SEED, reshuffle_each_iteration=True)
        ds = ds.map(prepare_train, num_parallel_calls=AUTOTUNE)
    else:
        ds = ds.map(prepare_eval, num_parallel_calls=AUTOTUNE)

    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

In [ ]:
train_gen = make_dataset(train_df_balanced, training=True)
val_gen = make_dataset(val_df, training=False)
test_gen = make_dataset(test_df, training=False)

for images, labels in train_gen.take(1):
    print("Batch image shape:", images.shape)
    print("Batch label shape:", labels.shape)
    print("Batch label distribution:", np.unique(labels.numpy(), return_counts=True))


In [ ]:
import matplotlib.pyplot as plt
import cv2

samples = df_model.sample(6, random_state=42)

plt.figure(figsize=(12, 8))
for i, (_, row) in enumerate(samples.iterrows()):
    img = cv2.imread(row["path"])
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.subplot(2, 3, i + 1)
    plt.imshow(img)
    plt.title("Pneumonia" if row["label"] == 1 else "Normal")
    plt.axis("off")

plt.tight_layout()
plt.show()

# EfficientNet-B0 with staged fine-tuning

In [ ]:
loss_fn = tf.keras.losses.BinaryCrossentropy(label_smoothing=0.03)

metrics = [
    tf.keras.metrics.AUC(name="auc", curve="ROC"),
    tf.keras.metrics.AUC(name="pr_auc", curve="PR"),
    tf.keras.metrics.Precision(name="precision"),
    tf.keras.metrics.Recall(name="recall")
]


In [ ]:
base_model = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

base_model.trainable = False

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = GlobalAveragePooling2D()(x)
x = Dropout(0.35)(x)
output = Dense(
    1,
    activation="sigmoid",
    kernel_regularizer=tf.keras.regularizers.l2(1e-2)
)(x)

model = Model(inputs=inputs, outputs=output)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-2),
    loss=loss_fn,
    metrics=metrics
)

model.summary()


In [ ]:
callbacks = [
    EarlyStopping(
        monitor="val_auc",
        mode="max",
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_auc",
        mode="max",
        factor=0.3,
        patience=2,
        min_lr=1e-2,
        verbose=1
    ),
    ModelCheckpoint(
        "best_stage1.keras",
        monitor="val_auc",
        mode="max",
        save_best_only=True,
        verbose=1
    )
]


In [ ]:
history_head = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=15,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1
)


In [ ]:
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=loss_fn,
    metrics=metrics
)

callbacks_ft = [
    EarlyStopping(
        monitor="val_auc",
        mode="max",
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_auc",
        mode="max",
        factor=0.3,
        patience=2,
        min_lr=1e-3,
        verbose=1
    ),
    ModelCheckpoint(
        "best_finetuned.keras",
        monitor="val_auc",
        mode="max",
        save_best_only=True,
        verbose=1
    )
]

history_ft_stage1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,
    class_weight=class_weights,
    callbacks=callbacks_ft,
    verbose=1
)


In [ ]:
val_true = val_df["label"].values.astype(int)
val_prob = model.predict(val_gen, verbose=1).ravel()

precisions, recalls, thresholds = precision_recall_curve(val_true, val_prob)
f1_scores = 2 * precisions[:-1] * recalls[:-1] / (precisions[:-1] + recalls[:-1] + 1e-8)

best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]
best_val_f1 = f1_scores[best_idx]

print("Best threshold:", best_threshold)
print("Best validation F1:", best_val_f1)


In [ ]:
for t in [0.10, 0.20, float(best_threshold), 0.35, 0.50]:
    y_pred = (val_prob >= t).astype(int)
    print(f"\nThreshold = {t:.4f}")
    print(confusion_matrix(val_true, y_pred))
    print(classification_report(val_true, y_pred, digits=4, zero_division=0))


In [ ]:
print("Validation ROC-AUC:", roc_auc_score(val_true, val_prob))
print("Validation PR-AUC:", average_precision_score(val_true, val_prob))
print("Mean predicted probability:", val_prob.mean())
print("Min predicted probability:", val_prob.min())
print("Max predicted probability:", val_prob.max())
print("Predicted positives at best threshold:", int((val_prob >= best_threshold).sum()))


In [ ]:
test_loss, test_auc, test_pr_auc, test_precision, test_recall = model.evaluate(
    test_gen,
    verbose=1
)

print(f"loss: {test_loss:.4f}")
print(f"auc: {test_auc:.4f}")
print(f"pr_auc: {test_pr_auc:.4f}")
print(f"precision@0.5: {test_precision:.4f}")
print(f"recall@0.5: {test_recall:.4f}")


In [ ]:
y_true = test_df["label"].values.astype(int)
y_prob = model.predict(test_gen, verbose=1).ravel()
y_pred = (y_prob >= best_threshold).astype(int)

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(y_true, y_pred, digits=4, zero_division=0))


In [ ]:
def plot_history(histories, metric="auc"):
    plt.figure(figsize=(8, 5))
    for name, hist in histories.items():
        if metric in hist.history and f"val_{metric}" in hist.history:
            plt.plot(hist.history[metric], label=f"{name} train")
            plt.plot(hist.history[f"val_{metric}"], label=f"{name} val")
    plt.title(metric.upper())
    plt.xlabel("Epoch")
    plt.ylabel(metric.upper())
    plt.legend()
    plt.show()

histories = {
    "fine_tune": history_ft_stage1
}

for metric_name in ["loss", "auc", "pr_auc", "precision", "recall"]:
    plot_history(histories, metric_name)

In [ ]:
cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()

sensitivity = tp / (tp + fn + 1e-8)
specificity = tn / (tn + fp + 1e-8)
precision_best = precision_score(y_true, y_pred, zero_division=0)
recall_best = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_true, y_prob)
pr_auc = average_precision_score(y_true, y_prob)

metrics_df = pd.DataFrame({
    "Metric": ["Precision", "Recall / Sensitivity", "Specificity", "F1-score", "ROC-AUC", "PR-AUC", "Best Threshold"],
    "Value": [precision_best, recall_best, specificity, f1, roc_auc, pr_auc, best_threshold]
})
metrics_df


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from lime import lime_image
from skimage.segmentation import mark_boundaries

explainer = lime_image.LimeImageExplainer()



def predict_fn(images):
    images = np.array(images)
    return np.concatenate([1 - model.predict(images), model.predict(images)], axis=1)

In [ ]:
sample_path = val_df.sample(1)['path'].values[0]

img = Image.open(sample_path).convert("RGB").resize((224, 224))
img_array = np.array(img) / 255.0

explanation = explainer.explain_instance(
    img_array.astype('double'),
    predict_fn,
    top_labels=1,
    hide_color=0,
    num_samples=1000
)

In [ ]:
temp, mask = explanation.get_image_and_mask(
    explanation.top_labels[0],
    positive_only=True,
    num_features=10,
    hide_rest=False
)

plt.figure(figsize=(6,6))
plt.imshow(mark_boundaries(temp, mask))
plt.title("LIME Explanation")
plt.axis("off")
plt.show()

In [ ]:
orig_pred = model.predict(np.expand_dims(img_array, axis=0))[0][0]
print("Original prediction:", orig_pred)


# keep only important regions
important_only = img_array.copy()
important_only[mask == 0] = 0

important_pred = model.predict(np.expand_dims(important_only, axis=0))[0][0]

print("Prediction with important regions:", important_pred)


# remove important regions
removed = img_array.copy()
removed[mask == 1] = 0

removed_pred = model.predict(np.expand_dims(removed, axis=0))[0][0]

print("Prediction without important regions:", removed_pred)

decision_impact = abs(orig_pred - removed_pred)
confidence_impact = abs(orig_pred - important_pred)

print("Decision Impact:", decision_impact)
print("Confidence Impact:", confidence_impact)

In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Normal", "Pneumonia"],
    yticklabels=["Normal", "Pneumonia"]
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix at Best Threshold")
plt.show()

fpr, tpr, _ = roc_curve(y_true, y_prob)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"ROC-AUC = {roc_auc:.4f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

prec_curve, rec_curve, _ = precision_recall_curve(y_true, y_prob)
plt.figure(figsize=(6, 5))
plt.plot(rec_curve, prec_curve, label=f"PR-AUC = {pr_auc:.4f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.legend()
plt.show()

print(metrics_df)


In [ ]:
last_conv_layer_name = None
for layer in reversed(base_model.layers):
    if isinstance(layer, tf.keras.layers.Conv2D):
        last_conv_layer_name = layer.name
        break

print("Last conv layer:", last_conv_layer_name)


In [ ]:
def get_gradcam_heatmap(model, img_array, last_conv_layer_name="top_conv"):
    base_model = None
    top_start_idx = None

    for i, layer in enumerate(model.layers):
        if isinstance(layer, tf.keras.Model):
            base_model = layer
            top_start_idx = i + 1
            break

    if base_model is None:
        raise ValueError("Backbone model not found inside model.")

    last_conv_layer = base_model.get_layer(last_conv_layer_name)

    conv_model = tf.keras.Model(
        inputs=base_model.input,
        outputs=last_conv_layer.output
    )

    classifier_input = tf.keras.Input(shape=last_conv_layer.output.shape[1:])
    x = classifier_input

    passed_last = False
    for layer in base_model.layers:
        if layer.name == last_conv_layer_name:
            passed_last = True
            continue
        if passed_last:
            x = layer(x)

    for layer in model.layers[top_start_idx:]:
        x = layer(x)

    classifier_model = tf.keras.Model(classifier_input, x)

    with tf.GradientTape() as tape:
        inputs = img_array
        conv_outputs = conv_model(inputs, training=False)
        tape.watch(conv_outputs)
        predictions = classifier_model(conv_outputs, training=False)
        loss = predictions[:, 0]

    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]
    heatmap = tf.reduce_sum(conv_outputs * pooled_grads, axis=-1)

    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

In [ ]:
def load_image_for_model(img_path, size=(224, 224)):
    img = tf.io.read_file(img_path)
    img = tf.image.decode_png(img, channels=1)
    img = tf.image.resize(img, size)
    img = tf.image.grayscale_to_rgb(img)
    img = tf.cast(img, tf.float32)
    img = preprocess_input(img)
    return tf.expand_dims(img, axis=0)

pneumonia_df = test_df[test_df["label"] == 1]

sample_row = pneumonia_df.sample(2, random_state=SEED).iloc[0]
img_path = sample_row["path"]

img_array = load_image_for_model(img_path, size=(IMG_SIZE, IMG_SIZE))
heatmap = get_gradcam_heatmap(model, img_array, last_conv_layer_name)

img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
img_rgb = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)

heatmap = cv2.resize(heatmap, (IMG_SIZE, IMG_SIZE))
heatmap_uint8 = np.uint8(255 * heatmap)
heatmap_color = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)

superimposed = cv2.addWeighted(img_rgb, 0.6, heatmap_color, 0.4, 0)

plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.imshow(img_rgb, cmap="gray")
plt.title("Original Pneumonia")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(heatmap_color)
plt.title("Grad-CAM")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(superimposed)
plt.title("Overlay")
plt.axis("off")

plt.show()



save_path = "outputs/gradcam_pneumonia.png"
plt.savefig(save_path, dpi=300, bbox_inches="tight")
print("Saved to:", save_path)

plt.show()

# ===== Prediction =====
pred = model.predict(img_array, verbose=0)

if pred.shape[-1] == 1:  # sigmoid
    pred_prob = float(pred[0][0])
    pred_label = int(pred_prob >= best_threshold)
else:  # softmax
    pred_prob = float(pred[0][1])
    pred_label = int(np.argmax(pred[0]))

print("True label:", "Pneumonia")
print("Predicted probability:", round(pred_prob, 4))
print("Predicted label:", "Pneumonia" if pred_label == 1 else "Normal")
print("Threshold used:", best_threshold)